In [1]:
#from pathlib import Path
#import sys
#sys.path.insert(0, str(Path.cwd().parent))
from rl_mind.core import Action
from rl_mind.data import Transitions
from agents import mlp, DetActor, Critic, DDPG
import torch


print(mlp([8,256,256,2],layer_norm=True))
print(mlp([8,256,256,2]) (torch.randn(5,8)).shape)

Sequential(
  (0): Linear(in_features=8, out_features=256, bias=True)
  (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
  (2): ReLU()
  (3): Linear(in_features=256, out_features=256, bias=True)
  (4): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
  (5): ReLU()
  (6): Linear(in_features=256, out_features=2, bias=True)
)
torch.Size([5, 2])


In [2]:
actor = DetActor(8, 2, layer_norm=True)
B = 5
obs = torch.randn(B, 8)

# obs is dim (5,8) so batch of 5
# actor takes input dim 8 and returns action dim 2

print(actor(obs).value.shape)   # torch.Size([5, 2]) bc B=5 and its an action of dim 2 
print(actor.act(obs).shape)     # same
print(actor.act(obs).abs().max() <= 1)   # tensor(True)
print(torch.equal(actor.act(obs), actor.act(obs)))   # True bc deterministic
print(torch.equal(actor(obs).value, actor(obs).value))   # False bc noisy

torch.Size([5, 2])
torch.Size([5, 2])
tensor(True)
True
False


In [3]:
critic = Critic(8, 2, layer_norm=True)
obs, act = torch.randn(B, 8), torch.rand(B, 2) * 2 - 1

q = critic(obs, act)
print(q.shape) # torch.Size([5])
print(q.requires_grad) # True
q

torch.Size([5])
True


tensor([ 0.1701, -0.1439, -0.3098, -0.1330, -0.5718],
       grad_fn=<SqueezeBackward1>)

In [4]:
agent = DDPG(8, 2, layer_norm=True)

# targets start identical to the online networks
same = all(torch.equal(p, q) for p, q in
           zip(agent.critic.parameters(), agent.critic_targ.parameters()))
print(same)   # expect True

# but they are separate objects
print(agent.critic is agent.critic_targ)   # expect False

True
False


In [5]:
agent = DDPG(8, 2, layer_norm=True)
B = 32
batch = Transitions(
    obs=torch.randn(B, 8),
    action=Action(value=torch.rand(B, 2) * 2 - 1),
    reward=torch.randn(B),
    next_obs=torch.randn(B, 8),
    terminated=torch.zeros(B, dtype=torch.bool),
)

before = [p.clone() for p in agent.actor.parameters()]
agent.update(batch)
after = list(agent.actor.parameters())

print(any(not torch.equal(b, a) for b, a in zip(before, after)))   # True

True


In [6]:
import numpy as np
from agents import DetActor
from bias import rollout

actor = DetActor(8, 2)
o, a, r, term = rollout(actor, seed=0)
print(o.shape, a.shape, r.shape, term)   # (T, 8) (T, 2) (T,) and True/False

o2, _, r2, _ = rollout(actor, seed=0)
print(np.allclose(r, r2))                

(108, 8) (108, 2) (108,) True
True
